In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import scienceplots
from utils_ising import visualize_ising, ising_2pt_corr_direction, ising2d_ham, ising2d_swendsen_wang
from model import get_rope_vit_model, ExponentialMovingAverage
from utils_train import rnd
from utils import compute_ess


TABLEAU_COLORS = {
    'blue': '#1f77b4',
    'orange': '#ff7f0e',
    'green': '#2ca02c',
    'red': '#d62728',
    'purple': '#9467bd',
    'brown': '#8c564b',
    'pink': '#e377c2',
    'gray': '#7f7f7f',
    'olive': '#bcbd22',
    'cyan': '#17becf'
}


## Model Loading

In [ ]:
L = 24
D = L**2
device = 'cuda:0'
J = 1
beta = 0.28 # high: 0.28, crit: 0.4407, low: 0.6
ckpt_dir = "checkpoints/L_16_ising/ising_high.pth"

# For 24x24, set hidden_size = 128, cond_dim = 128
cfg = {'tokens':2,
       'model':{'name': 'small_radd','type': 'ddit_wot','hidden_size': 64,
                'cond_dim': 64,'length': D,'n_blocks': 6,'n_heads': 4, 
                'dropout': 0.0,'use_checkpoint': False,'dtype': 'bfloat16'},
       'grad_clip': False, 'gradnorm_clip':1,'num_epochs': 1000, 'resample_every_n_step':10,
       'warmup_steps':0,
       'num_accum_steps':1,'batch_size': 256,'copy_flag_temp':None,'truncate_steps':32,
       'truncate_kl':False,'total_num_steps':32,'gumbel_temp':1,'gumbel_temp_schedule':'const', 
       'gumbel_temp_min': 0.1, 'eval_every':20, 'eval_batch_size': 32,
       'loss_fn':'wdce', 'wdce_num_replicates': 8, 'seed': None}


model = get_rope_vit_model(L, vocab_size=cfg['tokens'] + 1, embed_dim=cfg['model']['hidden_size'], depth=cfg['model']['n_blocks'], num_heads=cfg['model']['n_heads'], dtype=cfg['model']['dtype'], device=device)
ema = ExponentialMovingAverage(model.parameters(), decay=0.9999)
print('Model: num of params: {}, size: {:.2f} MB'.format(
    sum(p.numel() for p in model.parameters()),
    sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 ** 2)))

checkpoint = torch.load(ckpt_dir, map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
ema.load_state_dict(checkpoint['ema_state_dict'])
ema.store(model.parameters())
ema.copy_to(model.parameters())
model.eval()

## Generate Samples and Compute ESS

In [ ]:
log_rnd_list = []
samples_list = []
reward = lambda S: -beta * ising2d_ham(2 * S - 1, J = J, h = 0)

## Generate in total 4096 samples
with torch.no_grad():
    for i in range(4):
        samples, log_rnd = rnd(model, reward, 1024, 1, device)
        log_rnd_list.append(log_rnd)
        samples_list.append(samples)
        
log_rnd_list = torch.cat(log_rnd_list, dim=0)
samples_list = torch.cat(samples_list, dim=0)
ess = compute_ess(log_rnd_list)
print("Effective Sample Size: ", ess)


# Ground Truth samples with Swendsen-Wang for comparison
# swen_samples = ising2d_swendsen_wang(L = L, J = 1, beta = beta,  B = 128, num_collect = 32, burn_in = 2 ** 10, collect_every = 128, init = None)

## Visualization of Generated Samples

In [ ]:
fig = visualize_ising(samples_list[:25], 5, 5)

## Visualization of 2pt Correlation

In [ ]:
mdns_corr_x = np.array([ising_2pt_corr_direction(samples_list, r_x = r, r_y = 0, use_x = True, use_y = False).cpu().numpy() for r in range(-12,12)])
# swen_corr_x = np.array([ising_2pt_corr_direction(swen_samples, r_x = r, r_y = 0, use_x = True, use_y = False) for r in range(-L//2, L//2)])

print(mdns_corr_x)
color_4 = TABLEAU_COLORS['red'] # "#eb6936"  # 

    # Create figure with reduced spacing
fig, ax = plt.subplots(nrows = 1, ncols = 1, figsize=(4, 4))

# First row of plots
ax.plot(range(-12,12), mdns_corr_x, color = color_4, linestyle='--', marker = ">", label='MDNS')

ax.set_xticks(np.arange(-L//2,L//2, 2))
ax.set_xlabel(r'Distance $r$', fontsize = 14)
ax.set_ylabel(r'2-point Correlation', fontsize = 14)
ax.set_title("Two-point Correlation of Ising Model", fontsize = 14)

# Adjust the layout to be even tighter
plt.subplots_adjust(wspace=0.1, hspace=0.1, bottom = 0.2)
plt.tight_layout()

plt.show()
